<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_MLR/17_2_4_4_nested_cv_notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Evolution of Model Evaluation: Why Nested Cross-Validation?

If you're feeling a little dizzy from the way we keep changing how we split our data, you are not alone! 

First, we told you to use a **Train/Test split**. 
Then, we said, "Wait, use **Cross-Validation on the training data, plus a Test split**." 
Now, we are talking about ditching the single test split entirely in favor of **Nested Cross-Validation**. 

Why do we keep moving the goalposts? The short answer: **Hyperparameter tuning changes everything.** 

Let's walk through this progression using a concrete analogy: **Studying for a Final Exam.** In this analogy:
* **Training Data** = Your textbook and homework.
* **Validation Data** = Practice exams.
* **Test Data** = The actual Final Exam.
* **Hyperparameter Tuning** = Changing your study strategy (e.g., trying flashcards vs. re-reading vs. group study) to see what gets you the best score on the practice exams.

## Stage 1: The Simple Train / Test Split
When we first learned Machine Learning, we split our data once. 

**The Analogy:** You read the textbook (Train), and then you take the final exam (Test) to see how well you learned.

**The Visual:**
```text
Whole Dataset:
[==================================================]
[        Training Data (80%)       ][  Test (20%)  ]
```

**The Problem:** What if you were just incredibly lucky? What if the 20% of data that ended up in the Test set happened to be the exact chapters you studied the hardest? Or what if it was the one chapter you skipped? 

Because we only test *once*, our evaluation of the model is highly dependent on how the data was randomly split. We have high **variance** in our evaluation.

## Stage 2: Cross-Validation + Held-Out Test Set
To fix the "lucky/unlucky split" problem, we introduced K-Fold Cross Validation. But because we still need a final, unbiased evaluation, we lock away a Test set at the very beginning.

**The Analogy:** You lock the Final Exam (Test set) in a drawer. You use the textbook (Training Data), but you divide it into chapters. You study Chapter 1-4, and take a practice exam on Chapter 5. Then you study 2-5, and take a practice exam on Chapter 1. 

Crucially, **this is where we do Hyperparameter Tuning.** We try different study strategies (Flashcards, Highlighting, etc.). We pick the strategy that gives us the highest *average* score across all the practice exams. Finally, we take the real Final Exam.

**The Visual:**
```text
Whole Dataset:
[==================================================]
[        Training Data (80%)       ][  Test (20%)  ]
 |                                |
 |--- CV Fold 1: [Val][Trn][Trn]  |
 |--- CV Fold 2: [Trn][Val][Trn]  |
 |--- CV Fold 3: [Trn][Trn][Val]  |
```

**The Problem:** This is actually a *great* approach if you have millions of rows of data. But if your dataset is smaller (e.g., 500 rows), we run into two massive problems:

1. **Wasted Data:** We locked 20% of our precious data in a drawer. The model never got to train on it before the final evaluation!
2. **Overfitting to the Validation Set:** If you tune 1,000 different combinations of hyperparameters, you are eventually going to find a combination that performs amazingly on the Cross-Validation folds purely by chance. You have inadvertently "peeked" at the validation data so many times that your model is biased toward it. 
3. **The Unlucky Test Set Returns:** Because we tuned so heavily, the locked-away Test set is our *only* true measure of performance. But wait... what if that single Test set is an unlucky split? We are right back to the Stage 1 problem!

## Stage 3: Nested Cross-Validation (The Ultimate Solution)
How do we evaluate a heavily-tuned model without locking away 20% of our data forever? **Nested Cross Validation.**

Instead of having a single Test set, we use Cross-Validation to create our Test sets (the **Outer Loop**), and *inside* of that, we use Cross-Validation again to tune our hyperparameters (the **Inner Loop**).

**The Analogy:** 
You have 5 different Final Exams. 
For Final Exam #1, you use the remaining material to run practice tests, find your best study strategy, and then take Final Exam #1.
Then, you wipe your memory.
For Final Exam #2, you use the *other* material to run practice tests, find your best study strategy (it might be a different strategy this time!), and then take Final Exam #2.

**The Visual:**
```text
OUTER FOLD 1 (Testing the Process):
[        Inner Loop (Tuning)       ][ Outer Test 1 ]
[Val][Trn][Trn] -> best params -> Evaluate on Test 1

OUTER FOLD 2 (Testing the Process):
[ Outer Test 2 ][        Inner Loop (Tuning)       ]
Evaluate on Test 2 <- best params <- [Trn][Val][Trn]

OUTER FOLD 3 (Testing the Process):
[ Inner Loop (Tuning)  ][ Outer Test 3 ][Inner Loop]
[Trn][Trn][Val] -> best params -> Evaluate on Test 3
```

### Why is this so powerful?
Notice what happens in Nested CV: **Every single data point gets to be in a Test set exactly once.** 

We are no longer evaluating a *single model*. We are evaluating our **Model Building Process**. We are proving: "If I take a subset of data, tune hyperparameters, and test it on unseen data, how well does that overall process work on average?"

Once we prove the *process* works via Nested CV, we can confidently train one final model on **100% of our data**, tuning the hyperparameters one last time, and deploy it to the real world.

Let's look at how incredibly simple this is to code in `scikit-learn`.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier

# Load a relatively small dataset
X, y = load_breast_cancer(return_X_y=True)

# Define our model and the hyperparameters we want to tune
model = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10]
}

# ---------------------------------------------------------
# STAGE 2: Standard CV + Test Split (What we used to do)
# ---------------------------------------------------------
# We define an Inner CV for hyperparameter tuning.
# If we stopped here, we'd have to use train_test_split first and 
# lock away some data to evaluate this GridSearchCV.
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

clf = GridSearchCV(estimator=model, param_grid=param_grid, cv=inner_cv)

# ---------------------------------------------------------
# STAGE 3: NESTED CV (The Magic Trick)
# ---------------------------------------------------------
# We define an Outer CV for evaluating the entire tuning process.
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# THE MAGIC: We pass the GridSearchCV object (which contains the inner CV) 
# into cross_val_score (which acts as the outer CV).
# Scikit-learn automatically handles the nested looping!
nested_scores = cross_val_score(clf, X=X, y=y, cv=outer_cv)

print("Nested CV Scores for each Outer Fold:")
for i, score in enumerate(nested_scores):
    print(f"Outer Fold {i+1}: {score:.4f}")
    
print(f"\nUnbiased Estimate of Model Performance: {nested_scores.mean():.4f} (+/- {nested_scores.std():.4f})")

### Summary: When to use what?

1. **Train/Test Split:** Use for very quick and dirty baselines, or if you have an absolutely massive dataset (Millions of rows) where "lucky splits" are statistically impossible.
2. **CV + Held-out Test Set:** Use when you have a medium-to-large dataset, and you are only doing light hyperparameter tuning. 
3. **Nested CV:** Use when you have a small dataset (e.g., medical data, thousands of rows or less) AND you are testing many different models or doing heavy hyperparameter tuning. It prevents data leakage and ensures your final performance metric is mathematically sound.